In [21]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Annotated, List

import random

import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

from geneticengine.grammar.metahandlers.ints import IntRange
from geneticengine.grammar import extract_grammar
from geneticengine.grammar.decorators import weight
from geneticengine.problems import SingleObjectiveProblem, MultiObjectiveProblem
from geneticengine.random.sources import NativeRandomSource
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.evaluation.budget import TimeBudget, EvaluationBudget
from geneticengine.representations.tree.initializations import MaxDepthDecider, FullDecider, ProgressivelyTerminalDecider, PositionIndependentGrowDecider
from geneticengine.representations.tree.operators import GrowInitializer, PositionIndependentGrowInitializer, FullInitializer, RampedHalfAndHalfInitializer
from geneticengine.algorithms.gp.operators.initializers import HalfAndHalfInitializer, StandardInitializer
from geneticengine.representations.tree.treebased import TreeBasedRepresentation
from geneticengine.representations.grammatical_evolution.structured_ge import StructuredGrammaticalEvolutionRepresentation
from geneticengine.evaluation.recorder import CSVSearchRecorder
from geneticengine.evaluation.tracker import ProgressTracker
from geneticengine.evaluation.parallel import ParallelEvaluator

from geneticengine.algorithms.gp.operators.combinators import ParallelStep, SequenceStep
from geneticengine.algorithms.gp.operators.crossover import GenericCrossoverStep
from geneticengine.algorithms.gp.operators.elitism import ElitismStep
from geneticengine.algorithms.gp.operators.mutation import GenericMutationStep
from geneticengine.algorithms.gp.operators.novelty import NoveltyStep
from geneticengine.algorithms.gp.operators.selection import LexicaseSelection, TournamentSelection

from geneticengine.solutions.individual import Individual, PhenotypicIndividual
from geneticengine.algorithms.gp.structure import GeneticStep
from geneticengine.problems import Problem
from geneticengine.random.sources import RandomSource
from geneticengine.representations.api import RepresentationWithCrossover, Representation
from geneticengine.evaluation import Evaluator
from typing import Iterator, Any, TypeVar

from sklearn.datasets import load_breast_cancer

import pandas as pd

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_auc_score, roc_curve, auc
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

import time

import seaborn as sns
import matplotlib.pyplot as plt

import os     

from functools import lru_cache

import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold

In [22]:
import warnings
warnings.filterwarnings('ignore')

In [23]:
model_used = lgb.LGBMClassifier(n_estimators=50, max_depth=7, learning_rate=0.03, num_leaves=10, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1)
target_fpr_value = 0.05

### Functions and Data Preprocessing

In [24]:
df_orig = pd.read_csv('base.csv')

df_orig.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 32 columns):
 #   Column                            Non-Null Count    Dtype  
---  ------                            --------------    -----  
 0   fraud_bool                        1000000 non-null  int64  
 1   income                            1000000 non-null  float64
 2   name_email_similarity             1000000 non-null  float64
 3   prev_address_months_count         1000000 non-null  int64  
 4   current_address_months_count      1000000 non-null  int64  
 5   customer_age                      1000000 non-null  int64  
 6   days_since_request                1000000 non-null  float64
 7   intended_balcon_amount            1000000 non-null  float64
 8   payment_type                      1000000 non-null  object 
 9   zip_count_4w                      1000000 non-null  int64  
 10  velocity_6h                       1000000 non-null  float64
 11  velocity_24h                      1000

In [25]:
df_orig.head(2)

,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0


In [26]:
#save the df with the values until month 6
df = df_orig
#train until month 6 and test after month 6
train_df = df[df['month'] < 5].sample(frac=1, random_state=42)
val_df = df[df['month'] == 5].sample(frac=1, random_state=42)
test_df = df[df['month'] >= 6].sample(frac=1, random_state=42)
train_df.drop('month', axis=1, inplace=True)
test_df.drop('month', axis=1, inplace=True)
val_df.drop('month', axis=1, inplace=True)

#split into X and y
X_train = train_df.drop('fraud_bool', axis=1)
y_train = train_df['fraud_bool']
X_val = val_df.drop('fraud_bool', axis=1)
y_val = val_df['fraud_bool']
X_test = test_df.drop('fraud_bool', axis=1)
y_test = test_df['fraud_bool']
print(len(df))

1000000


In [27]:
# X = df.drop(['fraud_bool'], axis=1)
# y = df['fraud_bool']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

In [28]:
categorical_features = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os",
]


encoders = {}
for feat in categorical_features:
    encoder = LabelEncoder()
    X_train[feat] = encoder.fit_transform(X_train[feat])
    X_val[feat] = encoder.transform(X_val[feat])
    X_test[feat] = encoder.transform(X_test[feat])
    encoders[feat] = encoder

In [29]:
print(y_train.value_counts(),y_val.value_counts(), y_test.value_counts())

fraud_bool
0    668926
1      6740
Name: count, dtype: int64 fraud_bool
0    117912
1      1411
Name: count, dtype: int64 fraud_bool
0    202133
1      2878
Name: count, dtype: int64


### Baseline Model

In [30]:
feature_names = X_train.columns.tolist()
n_features = len(feature_names)

In [31]:
model_baseline = lgb.LGBMClassifier(n_estimators=50, max_depth=7, learning_rate=0.03, num_leaves=10, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1)

model_baseline.fit(X_train, y_train)

train_probs = model_baseline.predict_proba(X_train)[:,1]

fpr, tpr, thresholds = roc_curve(y_train, train_probs)

target_fpr = target_fpr_value

if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0]
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    train_tpr_at_fpr = tpr[best_index]

val_probs = model_baseline.predict_proba(X_val)[:,1]

fpr, tpr, thresholds = roc_curve(y_val, val_probs)

baseline_tpr_at_fpr = 0.0

if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0]
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    baseline_tpr_at_fpr = tpr[best_index]

print(f"Train TPR: {train_tpr_at_fpr}, Validation TPR: {baseline_tpr_at_fpr}")

Train TPR: 0.4946587537091988, Validation TPR: 0.4861800141743444


### Grammar

In [32]:
@dataclass
class Value(ABC):
    def evaluate(self):
        pass

class Scalar(ABC):
    pass

In [33]:
@weight(1.2)
@dataclass #Scalar Features (1)
class ScalarVar(Scalar): 
    index: Annotated[int, IntRange(0,n_features-1)]

    def evaluate(self, X_np):
        return X_np[:, self.index]
    
    def __str__(self):
        return feature_names[self.index]

In [34]:
#scalar -> scalar
@weight(0.2)
@dataclass 
class Add(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return self.left.evaluate(X_np) + self.right.evaluate(X_np)
    
    def __str__(self):
        return f"({self.left} + {self.right})"

@weight(0.2)
@dataclass
class Subtract(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return (self.left.evaluate(X_np)) - (self.right.evaluate(X_np))
    
    def __str__(self):
        return f"({self.left} - {self.right})"

@weight(0.2)
@dataclass
class Multiply(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return self.left.evaluate(X_np) * self.right.evaluate(X_np)
    
    def __str__(self):
        return f"({self.left} * {self.right})"

@weight(0.2)
@dataclass
class Divide(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        denom = self.right.evaluate(X_np)
        denom = np.where(denom == 0, 1e-6, denom)  # Avoid division by zero
        return self.left.evaluate(X_np) / denom
    
    def __str__(self):
        return f"({self.left} / {self.right})"
    
@weight(0.2)
@dataclass
class Sqrt(Scalar):
    value: Scalar

    def evaluate(self, X_np):
        val = self.value.evaluate(X_np)
        val = np.asarray(val)
        val = np.clip(val, a_min=0.0, a_max=None)
        return np.sqrt(val)

    def __str__(self):
        return f"sqrt({self.value})"
    
@weight(0.2)
@dataclass
class Log(Scalar):
    value: Scalar

    def evaluate(self, X_np):
        val = self.value.evaluate(X_np)
        val = np.asarray(val)
        val = np.where(val <= 0, 1e-6, val)
        return np.log(val)
    
    def __str__(self):
        return f"log({self.value})"

In [35]:
grammar = extract_grammar([Add, Subtract, Multiply, Divide, Sqrt, Log, ScalarVar], Scalar)
print(f"Grammar: {repr(grammar)}")

Grammar: Grammar<Starting=Scalar,Productions={
Scalar -> Add(right: Scalar, left: Scalar)<0.08>|
	Subtract(right: Scalar, left: Scalar)<0.08>|
	Multiply(right: Scalar, left: Scalar)<0.08>|
	Divide(right: Scalar, left: Scalar)<0.08>|
	Sqrt(value: Scalar)<0.08>|
	Log(value: Scalar)<0.08>|
	ScalarVar(index: Annotated[int])<0.50>
}


### Fitness and GP

In [36]:
ARCHIVE_TRAIN_DF = X_train.copy()
ARCHIVE_VAL_DF = X_val.copy()

ARCHIVE_TEMP : list[Individual] = []
ARCHIVE_IND : list[Individual] = []

X_train_np = ARCHIVE_TRAIN_DF.to_numpy()
X_val_np = ARCHIVE_VAL_DF.to_numpy()

In [37]:
def fitness_function(individual: Scalar): #individual -> expression
    
    if str(individual) in ARCHIVE_TRAIN_DF.columns:
        return [0.0, 0.0, 1000.0, 1000.0]
    
    start = time.perf_counter()
    train_feature = individual.evaluate(X_train_np)
    test_feature = individual.evaluate(X_val_np)
    if train_feature.ndim == 0:
        train_feature = np.full(X_train_np.shape[0], train_feature)
    if test_feature.ndim == 0:
        test_feature = np.full(X_val_np.shape[0], test_feature)

    X_train_augmented = np.c_[X_train_np, np.array(train_feature).reshape(-1,1)]
    X_val_augmented = np.c_[X_val_np, np.array(test_feature).reshape(-1,1)]

    model = lgb.LGBMClassifier(n_estimators=50, max_depth=7, learning_rate=0.03, num_leaves=10, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1)

    model.fit(X_train_augmented, y_train)
    
    probs = model.predict_proba(X_val_augmented)[:, 1]

    fpr, tpr, thresholds = roc_curve(y_val, probs)
    tpr_at_fpr = 0.0
    target_fpr = target_fpr_value
    if np.any(fpr <= target_fpr):
        valid_indices = np.where(fpr<=target_fpr)[0]
        best_indice = valid_indices[np.argmax(tpr[valid_indices])]
        tpr_at_fpr = tpr[best_indice]

    tpr_diff = tpr_at_fpr-baseline_tpr_at_fpr
        
    features, num_operations = analyse_complexity(individual)

    end = time.perf_counter()
    elapsed = end - start
    return [tpr_at_fpr, tpr_diff, num_operations, elapsed]



In [38]:
def analyse_complexity(individual: Scalar):
    if isinstance(individual, ScalarVar):
        return {individual.index}, 0 #unique feature
    
    total_features = set()
    total_operations = 1
    if hasattr(individual, 'left') and hasattr(individual, 'right'):
        left_features, left_operations = analyse_complexity(individual.left)
        right_features, right_operations = analyse_complexity(individual.right)
        total_features.update(left_features)
        total_features.update(right_features)
        total_operations += left_operations + right_operations
    elif hasattr(individual, 'arr'):
        arr_features, arr_operations = analyse_complexity(individual.arr)
        total_features.update(arr_features)
        total_operations += arr_operations
    return total_features, total_operations


In [ ]:
class ArchiveStep(GeneticStep):
    def iterate(
        self,
        problem: Problem,
        evaluator: Evaluator,
        representation: Representation,
        random: RandomSource,
        population: Iterator[PhenotypicIndividual],
        target_size: int,
        generation: int,
    ) -> Iterator[PhenotypicIndividual]:
        global ARCHIVE_TEMP, train_tpr_at_fpr, baseline_tpr_at_fpr, ARCHIVE_TRAIN_DF, ARCHIVE_VAL_DF, X_train_np, X_val_np, ARCHIVE_IND
        target_fpr = target_fpr_value
        for i, individual in enumerate(population):
            if individual.get_fitness(problem).fitness_components[0] > baseline_tpr_at_fpr:
                ARCHIVE_TEMP.append(individual)
                print("New Individual:", str(individual.get_phenotype()), "Fitness:", individual.get_fitness(problem).fitness_components)
            yield individual

        validated_individuals = []
        if ARCHIVE_TEMP:
            X_combined = np.vstack((X_train_np, X_val_np))
            y_combined = np.concatenate((y_train, y_val))
            skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
            print(f"Archive Size: {len(ARCHIVE_TEMP)}")
            for ind in ARCHIVE_TEMP:
                fold_tpr_scores = []

                combined_feature = ind.get_phenotype().evaluate(X_combined)
                if combined_feature.ndim == 0:
                    combined_feature = np.full(X_combined.shape[0], combined_feature)
                X_combined_augmented = np.c_[X_combined, np.array(combined_feature).reshape(-1,1)]

                for train_index, val_index in skf.split(X_combined_augmented, y_combined):
                    X_fold_train, X_fold_val = X_combined_augmented[train_index], X_combined_augmented[val_index]
                    y_fold_train, y_fold_val = y_combined[train_index], y_combined[val_index]

                    model = lgb.LGBMClassifier(n_estimators=50, max_depth=7, learning_rate=0.03, num_leaves=10, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1)
                    model.fit(X_fold_train, y_fold_train)

                    probs = model.predict_proba(X_fold_val)[:, 1]

                    fpr, tpr, thresholds = roc_curve(y_fold_val, probs)
                    fold_tpr_at_fpr = 0.0
                    if np.any(fpr <= target_fpr):
                        valid_indices = np.where(fpr <= target_fpr)[0]
                        best_indice = valid_indices[np.argmax(tpr[valid_indices])]
                        fold_tpr_at_fpr = tpr[best_indice]
                    fold_tpr_scores.append(fold_tpr_at_fpr)
                avg_tpr = np.mean(fold_tpr_scores)
                if avg_tpr > baseline_tpr_at_fpr:
                    print(f"Validated Individual: {str(ind.get_phenotype())}, Avg TPR: {avg_tpr}")
                    validated_individuals.append(ind)
                else:
                    print(f"Rejected Individual: {str(ind.get_phenotype())}, Avg TPR: {avg_tpr}")
            if validated_individuals:
                for ind in validated_individuals:
                    ARCHIVE_IND.append(ind)
                    train_feature_new = ind.get_phenotype().evaluate(X_train_np)
                    val_feature_new = ind.get_phenotype().evaluate(X_val_np)
                    ARCHIVE_TRAIN_DF[str(ind)] = train_feature_new
                    ARCHIVE_VAL_DF[str(ind)] = val_feature_new

                ARCHIVE_TRAIN_DF = ARCHIVE_TRAIN_DF.loc[:, ~ARCHIVE_TRAIN_DF.columns.duplicated()]
                ARCHIVE_VAL_DF = ARCHIVE_VAL_DF.loc[:, ~ARCHIVE_VAL_DF.columns.duplicated()]

                ARCHIVE_TRAIN_DF.columns = [str(col) for col in ARCHIVE_TRAIN_DF.columns]
                ARCHIVE_VAL_DF.columns = [str(col) for col in ARCHIVE_VAL_DF.columns]

                model = lgb.LGBMClassifier(n_estimators=50, max_depth=7, learning_rate=0.03, num_leaves=10, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1)
                model.fit(ARCHIVE_TRAIN_DF, y_train)
                probs = model.predict_proba(ARCHIVE_VAL_DF)[:,1]
                fpr, tpr, thresholds = roc_curve(y_val, probs)
                if np.any(fpr <= target_fpr):
                    valid_indices = np.where(fpr <= target_fpr)[0]
                    best_index = valid_indices[np.argmax(tpr[valid_indices])]
                    baseline_tpr_at_fpr = tpr[best_index]
                print(f"Validation TPR: {baseline_tpr_at_fpr}, Validation shape: {ARCHIVE_VAL_DF.shape}")
            
                X_train_np = ARCHIVE_TRAIN_DF.to_numpy()
                X_val_np = ARCHIVE_VAL_DF.to_numpy()

            ARCHIVE_TEMP = []

In [40]:
def lexicase_step():
    return SequenceStep(
        ArchiveStep(),
        ParallelStep(
            [
                ElitismStep(),
                # NoveltyStep(),
                SequenceStep(
                    LexicaseSelection(epsilon=True),
                    # TournamentSelection(tournament_size=3),
                    GenericCrossoverStep(0.9),
                    GenericMutationStep(0.1),
                )
            ],
            # weights=[0.05, 0.05, 0.9]
            weights=[0.1, 0.9]
        ),
    )

prob = MultiObjectiveProblem(
    fitness_function=fitness_function,
    minimize=[False, False, True, True],
)
r = NativeRandomSource(123)
alg = GeneticProgramming(
    problem=prob,
    budget=TimeBudget(600),
    population_size=100,
    representation=TreeBasedRepresentation(grammar, MaxDepthDecider(r, grammar, 4)),
    random=r,
    step=lexicase_step(),
    tracker=ProgressTracker(
        prob,
        recorders=[CSVSearchRecorder(
            csv_path='output.csv', 
            problem=prob, 
            fields={
                    "Eval Time": lambda t,i,p: i.get_fitness(p).fitness_components[3],
                    "TPR Test": lambda t,i,p: i.get_fitness(p).fitness_components[0],
                    "TPR Test Diff": lambda t,i,p: i.get_fitness(p).fitness_components[1],
                    "Expression": lambda t, i, p: i.get_phenotype(),
                    "Num Operations": lambda t,i,p: i.get_fitness(p).fitness_components[2],
                    'Generation': lambda t,i,p: i.metadata["generation"]
                    },
            only_record_best_individuals=False)]
    )
    
)

solutions = alg.search()

New Individual: ((sqrt(velocity_24h) - has_other_cards) * device_os) Fitness: [np.float64(0.4875974486180014), np.float64(0.0014174344436569952), 3, 1.4459915999905206]
New Individual: (housing_status + log(sqrt(name_email_similarity))) Fitness: [np.float64(0.48688873139617295), np.float64(0.0007087172218285254), 2, 1.1890619000187144]
New Individual: (customer_age + (email_is_free * velocity_24h)) Fitness: [np.float64(0.49043231750531535), np.float64(0.00425230333097093), 2, 1.1732400000328198]
New Individual: (credit_risk_score * customer_age) Fitness: [np.float64(0.4875974486180014), np.float64(0.0014174344436569952), 1, 1.5034637000062503]
New Individual: (phone_home_valid + log((current_address_months_count + bank_branch_count_8w))) Fitness: [np.float64(0.4897236002834869), np.float64(0.0035435861091424603), 2, 1.0995832000044174]
New Individual: (sqrt(phone_home_valid) / employment_status) Fitness: [np.float64(0.48688873139617295), np.float64(0.0007087172218285254), 2, 1.11002849

In [41]:
#train on the original set 
model = lgb.LGBMClassifier(n_estimators=350, max_depth=14, learning_rate=0.03, num_leaves=17, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1)
model.fit(X_train, y_train)
original_probs = model.predict_proba(X_test)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, original_probs)
target_fpr = target_fpr_value
if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0]
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    original_tpr_at_fpr = tpr[best_index]
print(f"Original Test TPR at FPR {target_fpr}: {original_tpr_at_fpr}")

#train on the augmented set
X_test_enhanced = X_test.copy()
for ind in ARCHIVE_IND:
    X_test_enhanced[str(ind.get_phenotype())] = ind.get_phenotype().evaluate(X_test.to_numpy())
model = lgb.LGBMClassifier(n_estimators=350, max_depth=14, learning_rate=0.03, num_leaves=17, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1)
model.fit(ARCHIVE_TRAIN_DF, y_train)
augmented_probs = model.predict_proba(X_test_enhanced)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, augmented_probs)
target_fpr = target_fpr_value
if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0]
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    augmented_tpr_at_fpr = tpr[best_index]
print(f"Augmented Test TPR at FPR {target_fpr}: {augmented_tpr_at_fpr}")
print(f"Improvement in TPR at FPR {target_fpr}: {augmented_tpr_at_fpr - original_tpr_at_fpr}")

# results_text = f"""
# FINAL RESULTS (on Test Set)
# {'='*60}
# Original (Baseline) TPR @ FPR={target_fpr_value}: {original_tpr_at_fpr:.4f}
# GP Enhanced TPR @ FPR={target_fpr_value}: {augmented_tpr_at_fpr:.4f}
# Improvement: {(augmented_tpr_at_fpr - original_tpr_at_fpr):+.4f}
# Number of engineered features: {len(ARCHIVE_IND)}
# {'='*60}

# Engineered Features:
# """

# for i, ind in enumerate(ARCHIVE_IND, 1):
#     results_text += f"\n{i}. {str(ind.get_phenotype())}"

# with open('results.txt', 'w') as f:
#     f.write(results_text)

# print(results_text)
# print("Results saved to results.txt")

Original Test TPR at FPR 0.05: 0.5482974287699791
Augmented Test TPR at FPR 0.05: 0.5399583043780403
Improvement in TPR at FPR 0.05: -0.008339124391938846


In [42]:
# df_new = df_orig.copy()

# categorical_features = [
#     "payment_type",
#     "employment_status",
#     "housing_status",
#     "source",
#     "device_os",
# ]
# for feat in categorical_features:
#     encoder = encoders[feat]
#     df_new[feat] = encoder.transform(df_new[feat])
# df_new_np = df_new.to_numpy()
# for ind in ARCHIVE_IND:
#     df_new[str(ind.get_phenotype())] = ind.get_phenotype().evaluate(df_new_np)

# df_new.columns


In [43]:
# df_new.to_csv('base_enhanced.csv', index=False)